**Logics for AI - Project 3 - Alessio Di Ubaldo**

**"Development of a classifier to predict GAD in depressed subjects: a Gender-Bias assessment"**

This notebook includes the implementation of the entire project. The aim is to develop a classifier trained on a real dataset and to observe how the model behaves while adding features, eventually assessing Group Fairness and Individual Fairness in relation to Gender. The main steps of this notebook can be summarized as following:

**1. Data Preparation**: importing libraries and dataset, filtering subjects by depressive symptoms (PHQ >= 10) and by age (16-80 years), setting the Target and defining the final sample.

**2. Model Training**: we start from a Baseline Model (only trained on gender) and then proceed adding other blocks of features (clinical scores and demographic variables).

**3. Removing features**: once all features have been added, their weights are analyzed and, with the support of the scientific literature, we decide which features are irrelevant.

**4. Assessment of Individual and Group Fairness**: through the implementation of Correction Distance and Jaccard Index, we can verify whether the model is gender-biased.



---



First of all, we **import the libraries and the modules** needed to carry out all the steps of the project: NumPy for the mathematical operations, Pandas to work on the dataframe, and the necessary modules from sklearn to train the model and check its accuracy.

**Logistic Regression** is the chosen learning function for our model. It is perfect to address the classification task of the project because, based on data in input, this algorithm outputs probabilities between 0 and 1, and uses a threshold (e.g., 0.5) to make the final prediction. In our scenario, given the set of features in input, the model will predict whether the subject has a Generalized Anxiety Disorder (GAD = 1) or not (GAD = 0).

The chosen **dataset** is a real public dataset from the paper "Temporal dynamics in psychological assessments: a novel dataset with scales and response times" (Zhao, 2023). It comprises data from 24,292 students answering to four recognized psychological scales (PHQ-9, GAD-7, ISI, and PSS), in addition to demographic variables such as gender, age, education and smoking / drinking habits. Further information about the dataset and a preview of each file is available at the following link: https://zenodo.org/records/10423537

After loading all the .csv files of the dataset, we use .shape to display the number of raws and columns for each file. Doing so, we can be sure that each file includes the same number of subjects.  

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Data loading
demo = pd.read_csv('demographic.csv')
phq9 = pd.read_csv('phq9.csv')
gad7 = pd.read_csv('gad7.csv')
pss  = pd.read_csv('pss.csv')
isi  = pd.read_csv('isi.csv')

# Check that each file has the same number of raws (subjects)
# .shape shows us both the number of raws and columns
print(f"Demographics: {demo.shape}")
print(f"PHQ-9:       {phq9.shape}")
print(f"GAD-7:       {gad7.shape}")
print(f"PSS:         {pss.shape}")
print(f"ISI:         {isi.shape}")



Demographics: (24292, 6)
PHQ-9:       (24292, 20)
GAD-7:       (24292, 16)
PSS:         (24292, 30)
ISI:         (24292, 16)




---



The dataset includes the specific answers of the subjects and the response time to each item of the clinical questionnaires, but for the purpose of the project these features can be ignored. **Only the final score** for each clinical scale is considered.

Then, we proceed to the creation of a **unique dataset**, by merging all files based on the id of the subjects, which is the same in each file.  

The below output returns:
*   the number of raws and columns for the resulting dataset
*   all the features (columns) included
*   a preview of the dataframe showing information of three different subjects   






In [ ]:
# Keep only the final score for each clinical scale
phq9_clean = phq9[['export_id', 'score']].rename(columns={'score': 'phq_score'})
gad7_clean = gad7[['export_id', 'score']].rename(columns={'score': 'gad_score'})
pss_clean  = pss[['export_id', 'score']].rename(columns={'score': 'pss_score'})
isi_clean  = isi[['export_id', 'score']].rename(columns={'score': 'isi_score'})

# Merging all dataframes using the id as common key
df = demo.merge(phq9_clean, on='export_id')
df = df.merge(gad7_clean,  on='export_id')
df = df.merge(pss_clean,   on='export_id')
df = df.merge(isi_clean,   on='export_id')

print(f"Unique dataset: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print("\nPreview:")
print(df.head(3))

Unique dataset: (24292, 10)

Columns: ['export_id', 'gender', 'age', 'edu', 'smoke', 'drink', 'phq_score', 'gad_score', 'pss_score', 'isi_score']

Preview:
   export_id  gender   age                edu         smoke  \
0      61793  female  19.0  bachelor's degree  never smokes   
1      61809  female  18.0  bachelor's degree  never smokes   
2      61737    male  40.0    master's degree  never smokes   

                                         drink  phq_score  gad_score  \
0                                 never drinks          0          0   
1                                 never drinks          0          0   
2  drinks occasionally (less than once a week)          0          0   

   pss_score  isi_score  
0          2          0  
1         21          0  
2          4          0  




---



Now we can start **filtering** the dataset.

The first fundamental step is the selection of the **individuals with significant depressive symptoms**. This is possible by keeping only the individuals who have a score of '10' or higher in the PHQ-9 scale.   

Another important filtering operation regards selecting subjects with an age comprised **between 16 and 80 years**. Indeed, we can notice within the dataset that there are subjects being 1 or 99 years old: these extreme values clearly represent errors in age reporting. Although excluding individuals with a certain age will reduce our final sample, such an issue cannot be ignored because 'age' might constitute a relevant property within our context.  

Next, we set the **binary Target**, namely, the actual GAD diagnosis for each patient. A score of '10' or higher in the GAD questionnaire is usually associated to the presence of a Generalized Anxiety Disorder, so we use this "threshold" to discriminate between patients with or without GAD.

Eventually, we **encode the 'gender**' variable to convert text categories into a binary numeric format.  

In [ ]:
# Selecting only subjects with significant depressive symptoms
df_dep = df[df['phq_score'] >= 10].copy()
print(f"Subjects with PHQ >= 10: {len(df_dep)}")

# Selecting only subjects with age 16-80
# (there are "age" values such as '1' or '99', which are cleary errors)
df_dep = df_dep[(df_dep['age'] >= 16) & (df_dep['age'] <= 80)].copy()
print(f"Subjects after filtering age: {len(df_dep)}")

# Binary Target: GAD = 1 if gad_score >= 10, otherwise 0
df_dep['GAD_binary'] = (df_dep['gad_score'] >= 10).astype(int)

# Encoding gender: male = 0, female = 1
df_dep['gender_encoded'] = df_dep['gender'].map({'male': 0, 'female': 1})

print(f"\nTarget Distribution GAD_binary:")
print(df_dep['GAD_binary'].value_counts())

print(f"\nGender distribution:")
print(df_dep['gender'].value_counts())

print(f"\nGAD rate by gender:")
print(df_dep.groupby('gender')['GAD_binary'].mean().round(3))



Subjects with PHQ >= 10: 1316
Subjects after filtering age: 1310

Target Distribution GAD_binary:
GAD_binary
0    899
1    411
Name: count, dtype: int64

Gender distribution:
gender
female    805
male      505
Name: count, dtype: int64

GAD rate by gender:
gender
female    0.320
male      0.303
Name: GAD_binary, dtype: float64


The above output returns some interesting information, such as the actual number of subjects (1310), the number of patients with GAD (411), the number of males and females, and the per-gender GAD rate.



---



Now we can conclude the **definition of our sample**. Specifically, we discard possible NaN values (although in the original paper is written that participants with missing values have already been excluded) taking into account all the features we are going to add step-by-step, until getting the final model.

The output actually tells us that no NaN values were present, since the number of subjects has not changed.   

In [ ]:
# SAMPLE DEFINITION

# Including all features needed for the final model
final_features = [
    'gender_encoded',
    'phq_score',
    'pss_score',
    'isi_score',
    'age',
    'edu',
    'smoke',
    'drink',
]

# Dropping Nan values in any feature or in the target
# By doing it now, each model will work on the same sample
df_clean = df_dep.dropna(subset=final_features + ['GAD_binary']).copy()

print(f"Subjects after dropping NaN values: {len(df_clean)}")

Subjects after dropping NaN values: 1310




---



**Baseline Model: Gender Only**

We move on to the training of the Baseline Model, which is only trained on our **protected attribute: Gender**. First of all, we define two variables:

*   **X_gender** represents the only feature in input that the model will use to make predictions, which is 'gender_encoded'. More in general, the variable X includes the set of features in input the model is trained on.

*   **y** is the target output ψ, i.e. the actual presence or absence of GAD for each subject, given by the feature 'GAD_binary'.


Next, data is divided into **Training test** and **Test set** through the train_test_split function. Typically, 80% of the dataset is used to train the model and the remaining 20% to observe the model's accuracy on unseen data. We adopt this partition.      


In [ ]:
# BASELINE MODEL (GENDER ONLY)

X_gender = df_clean[['gender_encoded']]
y = df_clean['GAD_binary']

# Split data into training (80%) and test (20%)
X_train, X_test, y_train, y_test = train_test_split(X_gender, y, test_size=0.2, random_state=42)

# Training
model_1 = LogisticRegression(random_state=42)
model_1.fit(X_train, y_train)

# Predictions
y_pred_1 = model_1.predict(X_test)

# Results
print("BASELINE MODEL - GENDER ONLY")
print(f"Overall accuracy: {accuracy_score(y_test, y_pred_1)*100:.2f}%")
print(f"\nCoefficients:")
print(f"Gender (coef):  {model_1.coef_[0][0]:.4f}\n")


# Accuracy by gender
for gend in ['female', 'male']:
    mask = df_clean['gender'] == gend
    X_g = df_clean.loc[mask, ['gender_encoded']]
    y_g = df_clean.loc[mask, 'GAD_binary']
    acc_g = accuracy_score(y_g, model_1.predict(X_g))
    n = mask.sum()
    print(f"Prediction accuracy for {gend}s: {acc_g *100:.2f}%")

BASELINE MODEL - GENDER ONLY
Overall accuracy: 70.61%

Coefficients:
Gender (coef):  0.0329

Prediction accuracy for females: 67.95%
Prediction accuracy for males: 69.70%


**Baseline Model results**:

* We can notice that the weight (coefficient) assigned to the variable 'gender'
is very low, meaning that gender alone is a weak predictor.

* Moreover, prediction accuracy is similar across males and females: the model is not returning different outputs based on gender  



---



**Model 2: Gender + PHQ score**

Now we add to the model the PHQ score: our first NPR attribute and, probably,  the **most important clinical predictor** of GAD in our set of features.

In [ ]:
# MODEL 2: GENDER + PHQ SCORE

# Features: gender + phq_score
X_phq = df_clean[['gender_encoded', 'phq_score']]
y = df_clean['GAD_binary']

X_train, X_test, y_train, y_test = train_test_split(X_phq, y, test_size=0.2, random_state=42)

model_2 = LogisticRegression(random_state=42)
model_2.fit(X_train, y_train)
y_pred_2 = model_2.predict(X_test)

print("MODEL 2 — Gender + PHQ Score")
print(f"Overall accuracy: {accuracy_score(y_test, y_pred_2)*100:.2f}%")
print(f"\nCoefficients:")
for fname, coef in zip(['gender_encoded', 'phq_score'], model_2.coef_[0]):
    print(f" - {fname}: {coef:.4f}")
print()

# Accuracy by gender
for gend in ['female', 'male']:
    mask = df_clean['gender'] == gend
    X_g = df_clean.loc[mask, ['gender_encoded', 'phq_score']]
    y_g = df_clean.loc[mask, 'GAD_binary']
    acc_g = accuracy_score(y_g, model_2.predict(X_g))
    n = mask.sum()
    print(f"Prediction accuracy for {gend}s: {acc_g*100:.2f}%")

MODEL 2 — Gender + PHQ Score
Overall accuracy: 77.10%

Coefficients:
 - gender_encoded: 0.2283
 - phq_score: 0.2918

Prediction accuracy for females: 74.16%
Prediction accuracy for males: 80.40%


**Model 2 results:**

* **Overall accuracy is increased** compared to baseline model, confirming that depressive symptoms have a strong impact in predicting GAD.

* The **coefficient of phq_score** is the highest, showing significant predictive weight.

* The **coefficient of gender_encoded** also increased compared to the baseline, and this deserves attention: gender is gaining predictive weight even though not clinically meaningful. Furthermore, a positive coefficient of gender means that **women are associated with a higher probability of being diagnosed with GAD** (since we encoded males = 0 and females = 1) .

* The **per-gender accuracy gap** is amplified, suggesting that PHQ score alone does not capture the clinical picture equally well for both sexes. The model is less accurate for female subjects when gender is the only demographic variable included.



---



**Model 3: adding clinical features**

We proceed with the addition of two clinical scales: '**pss-score**' investigates perceived stress, while '**isi_score**' is an index of insomnia.


In [ ]:
# MODEL 3: ADDING CLINICAL FEATURES

# Features: gender + phq_score + pss_score + isi_score
X_clinical = df_clean[['gender_encoded', 'phq_score', 'pss_score', 'isi_score']]
y = df_clean['GAD_binary']

X_train, X_test, y_train, y_test = train_test_split(X_clinical, y, test_size=0.2, random_state=42)

model_3 = LogisticRegression(random_state=42)
model_3.fit(X_train, y_train)
y_pred_3 = model_3.predict(X_test)

print("MODEL 3 — Gender + PHQ + PSS + ISI")
print(f"Overall accuracy: {accuracy_score(y_test, y_pred_3)*100:.2f}%")
print(f"\nCoefficients:")
feature_names = ['gender_encoded', 'phq_score', 'pss_score', 'isi_score']
for fname, coef in zip(feature_names, model_3.coef_[0]):
    print(f" - {fname}: {coef:.4f}")
print()

# Accuracy by gender
for gend in ['female', 'male']:
    mask = df_clean['gender'] == gend
    X_g = df_clean.loc[mask, ['gender_encoded', 'phq_score', 'pss_score', 'isi_score']]
    y_g = df_clean.loc[mask, 'GAD_binary']
    acc_g = accuracy_score(y_g, model_3.predict(X_g))
    n = mask.sum()
    print(f"Prediction accuracy for {gend}s: {acc_g *100:.2f}%")

MODEL 3 — Gender + PHQ + PSS + ISI
Overall accuracy: 75.95%

Coefficients:
 - gender_encoded: -0.0594
 - phq_score: 0.2287
 - pss_score: 0.1295
 - isi_score: 0.0719

Prediction accuracy for females: 77.52%
Prediction accuracy for males: 80.20%


**Model 3 results:**


**Gender coefficient went negative**: while in the previous model the gender coefficient had a positive value, now it switched to -0.0594, meaning that being a man is associated with slightly higher GAD probability.
This help us provide a possible explanation for the results of the previous model, where gender was probably **acting as a proxy** for stress and insomnia (possibly due to different PSS/ISI profiles between males and females, and gender was "absorbing" that signal). Once we directly add PSS and ISI, gender loses that proxy role and its actual independent contribution becomes close to zero and slightly negative.

These results show that gender was carrying hidden information about other variables, which is exactly how protected attributes introduce bias in classifiers.



---



**Encoding demographic features**

The next step of the project is to add all the demographic variables to our model. However, since these features include text categories, before defining and training the final model it is necessary to **assign a number to each category**, by following a logical order.

In [ ]:
# ENCODING DEMOGRAPHIC FEATURES

# All features including demographic variables
X_all = df_clean[['gender_encoded', 'phq_score', 'pss_score', 'isi_score', 'age', 'edu', 'smoke', 'drink']]
y = df_clean['GAD_binary']

# Since these features include text categories, let's display current values
print("edu values:", df_clean['edu'].unique())
print("\n smoke values:", df_clean['smoke'].unique())
print("\n drink values:", df_clean['drink'].unique())

# ENCODING PHASE
# EDU: encode by degree level
edu_map = {
    'associate degree': 0,
    "bachelor's degree": 1,
    'master\'s degree': 2,
    'doctorate degree': 3
}

# SMOKE: encode by smoking intensity
smoke_map = {
    'never smokes': 0,
    'occasional smoker (cumulative smoking <10 packs)': 1,
    'former smoker (cumulative smoking >10 packs), but not in the past year': 2,
    'current smoker (cumulative smoking >10 packs)': 3
}

# DRINK: encode by drinking frequency
drink_map = {
    'never drinks': 0,
    'drank in the past (more than once a week), but not in the past year': 1,
    'drinks occasionally (less than once a week)': 2,
    'current regular drinker (more than once a week)': 3
}

# Apply mappings to the dataframe
df_clean['edu_encoded']   = df_clean['edu'].map(edu_map)
df_clean['smoke_encoded'] = df_clean['smoke'].map(smoke_map)
df_clean['drink_encoded'] = df_clean['drink'].map(drink_map)



edu values: ["bachelor's degree" "master's degree" 'doctorate degree'
 'associate degree']

 smoke values: ['never smokes' 'current smoker (cumulative smoking >10 packs)'
 'occasional smoker (cumulative smoking <10 packs)'
 'former smoker (cumulative smoking >10 packs), but not in the past year']

 drink values: ['drinks occasionally (less than once a week)' 'never drinks'
 'current regular drinker (more than once a week)'
 'drank in the past (more than once a week), but not in the past year']




---



**Model 4: adding demographic features**

Now it is possible to add the block of encoded demographic variables: age, education, smoking and drinking frequencies.  

In [ ]:
# MODEL 4: ADDING ENCODED DEMOGRAPHIC FEATURES

# All features including encoded demographics
features_model_4 = ['gender_encoded', 'phq_score', 'pss_score', 'isi_score',
                    'age', 'edu_encoded', 'smoke_encoded', 'drink_encoded']

X_all = df_clean[features_model_4]
y = df_clean['GAD_binary']

X_train, X_test, y_train, y_test = train_test_split(X_all, y, test_size=0.2, random_state=42)

model_4 = LogisticRegression(random_state=42)
model_4.fit(X_train, y_train)
y_pred_4 = model_4.predict(X_test)

print("MODEL 4 — All Features")
print(f"Overall accuracy: {accuracy_score(y_test, y_pred_4)*100:.2f}%")
print(f"\nCoefficients:")
for fname, coef in zip(features_model_4, model_4.coef_[0]):
    print(f" - {fname}: {coef:.4f}")
print()

# Accuracy by gender
for gend in ['female', 'male']:
    mask = df_clean['gender'] == gend
    X_g = df_clean.loc[mask, features_model_4]
    y_g = df_clean.loc[mask, 'GAD_binary']
    acc_g = accuracy_score(y_g, model_4.predict(X_g))
    n = mask.sum()
    print(f"Prediction accuracy for {gend}s: {acc_g*100:.2f}%")

MODEL 4 — All Features
Overall accuracy: 75.95%

Coefficients:
 - gender_encoded: -0.0381
 - phq_score: 0.2272
 - pss_score: 0.1292
 - isi_score: 0.0720
 - age: 0.0641
 - edu_encoded: 0.0136
 - smoke_encoded: -0.0001
 - drink_encoded: 0.0319

Prediction accuracy for females: 78.14%
Prediction accuracy for males: 80.00%


**Model 4 results:**

* Overall accuracy remains **stable at 75.95%**, consistent with Model 3, indicating that the demographic variables are not actually adding meaningful predictive power.

* **Coefficient of gender** is still close to zero and has a negative value. This confirms that the gender effect observed in Model 2 was spurious, mainly driven by its correlation with PSS and ISI rather than a real clinical relationship with GAD.

* The **per-gender accuracy gap** is quite small, suggesting the full feature set can lead to a more equal classifier across sexes.

**Removing irrelevant features**

By analyzing the resulting coefficients from the training of Model 4, we are now able to identify irrelevant properties, meaning that we will formally assign *w = 0* to them.


* phq_score (0.2272) and pss_score (0.1292) are the **strongest predictors**: depression severity and perceived stress clinically correlate with GAD, thus showing a significant predictive weight.

* isi_score (0.0720) and age (0.0641) are weak values, but scientific literature supports the idea of these variables having a correlation with GAD (further details in the README).  

* edu_encoded (0.0136), drink_encoded (0.0319) and smoke_encoded (-0.0001) are near zero: we can consider these features as **clinically irrelevant** for GAD prediction.

In the following block of code, only relevant features are kept and the final model is defined:


In [ ]:
# FINAL MODEL - Removing irrelevant features

# Keeping: gender, phq, pss, isi, age
# Discarding: edu, smoke, drink

features_final = ['gender_encoded', 'phq_score','pss_score', 'isi_score', 'age']

X_final = df_clean[features_final]
y = df_clean['GAD_binary']

X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)

model_final = LogisticRegression(random_state=42)
model_final.fit(X_train, y_train)
y_pred_final = model_final.predict(X_test)

print("FINAL MODEL — Reduced Feature Set")
print(f"Overall accuracy: {accuracy_score(y_test, y_pred_final)*100:.2f}%")
print(f"\nCoefficients:")
for fname, coef in zip(features_final, model_final.coef_[0]):
    print(f" - {fname}: {coef:.4f}")
print()

# Accuracy by gender
for gend in ['female', 'male']:
    mask = df_clean['gender'] == gend
    X_g = df_clean.loc[mask, features_final]
    y_g = df_clean.loc[mask, 'GAD_binary']
    acc_g = accuracy_score(y_g, model_final.predict(X_g))
    n = mask.sum()
    print(f"Prediction accuracy for {gend}s: {acc_g*100:.2f}%")

FINAL MODEL — Reduced Feature Set
Overall accuracy: 75.57%

Coefficients:
 - gender_encoded: -0.0566
 - phq_score: 0.2271
 - pss_score: 0.1294
 - isi_score: 0.0726
 - age: 0.0647

Prediction accuracy for females: 78.26%
Prediction accuracy for males: 80.59%


**Final model results:**

With the overall accuracy of the final model still being around 75%, we can confirm that 'education', 'smoke' and 'drink' were irrelevant variables.

The rest of these results - coefficients and per-gender accuracy - is basically supporting our previous observations: all the included features are meaningful and allow the model to equally perform across gender.  





---



Now, we can move on to the last fundamental step: **the assessment of Individual and Group Fairness**.

To do so, we can apply **Correction Distance** to verify Group Fairness and the **Jaccard Index** to verify Individual Fariness.

**Correction Distance**

Correction Distance can only be obtained through a **continuous value** that allows us to measure how close or far is the model from the correct output (i.e., the Ground Truth ¬ψ, which in our framework is represented by the observed value of 'GAD_binary').

This is why, first of all, we need to get the predicted probability of the patients being diagnosed with GAD, rather than the binary prediction alone.

We use ***predict_proba*** and take the probabilities for the positive class (GAD=1). Then, we add to the dataframe three columns: the predicted probability of the model, the binary prediction label and the correct output.   

In [ ]:
# predict_proba returns the probability for each class [P(GAD=0), P(GAD=1)]
# Take the probability of the positive class (GAD=1)
proba_final = model_final.predict_proba(X_final)[:, 1]

# Adding to the dataframe
df_clean = df_clean.copy()
df_clean['pred_proba'] = proba_final
df_clean['pred_label'] = model_final.predict(X_final)
df_clean['correct'] = (df_clean['pred_label'] == df_clean['GAD_binary']).astype(int)

print("Preview:")
print(df_clean[['gender', 'GAD_binary', 'pred_proba',
                'pred_label', 'correct']].head(10))

Preview:
     gender  GAD_binary  pred_proba  pred_label  correct
19   female           1    0.168308           0        0
41   female           0    0.233108           0        1
81   female           0    0.069763           0        1
113  female           0    0.119179           0        1
120  female           1    0.331832           0        0
123  female           1    0.753157           1        1
132  female           0    0.375865           0        1
144    male           0    0.144002           0        1
146    male           0    0.214714           0        1
155  female           0    0.145313           0        1




---



Now we can do the calculations for the Correction Distance, implementing the following formula:

**C(S, ψ) = 1 - (N - M) * accuracy**

 where:
* N = 1 (total normalized weight)
* M = pred_proba if true_label=1, else (1 - pred_proba)
* accuracy = overall model accuracy




In [ ]:
# CORRECTION DISTANCE

# Overall model accuracy is needed for calculations
model_accuracy = accuracy_score(y_test, y_pred_final)
print(f"Model accuracy: {model_accuracy*100:.2f}%")

def correction_distance(true_label, pred_proba, accuracy):

    if true_label == 1:
        M = pred_proba          # probability already assigned to correct class
    else:
        M = 1 - pred_proba

    N = 1
    C = 1 - (N - M) * accuracy
    return C

# Apply to each subject
df_clean['correction_dist'] = df_clean.apply(
    lambda row:
        correction_distance(row['GAD_binary'],
        row['pred_proba'],
        model_accuracy
    ),
    axis=1
)

print("\nCorrection Distance — General Stats:")
print(df_clean['correction_dist'].describe().round(4))
print("\nCorrection Distance by gender:")
print(df_clean.groupby('gender')['correction_dist'].describe().round(4))

Model accuracy: 75.57%

Correction Distance — General Stats:
count    1310.0000
mean        0.7747
std         0.1856
min         0.2613
25%         0.6671
50%         0.8359
75%         0.9232
max         0.9983
Name: correction_dist, dtype: float64

Correction Distance by gender:
        count    mean     std     min     25%     50%     75%     max
gender                                                               
female  805.0  0.7640  0.1894  0.2613  0.6537  0.8228  0.9181  0.9934
male    505.0  0.7918  0.1783  0.2896  0.7121  0.8611  0.9307  0.9983




---



**Group Fairness**

Once we have the Correction Distance for each subject, we can move on to the implementation of the formula for Group Fairness:

**Fairness_group(S,T) = | C(S, GAD(a)) − C(S, GAD(b)) | < ε**

Group Fairness is obtained by getting the absolute difference between the mean Correction Distance of females and that of males, and the resulting value is then compared with a threshold ε (usually set at 0.05). If the observed value is smaller than the threshold, we can state that Group Fairness is satisfied.    

In [ ]:
# GROUP FAIRNESS

# Average Correction Distance by gender
cd_female = df_clean[df_clean['gender'] == 'female']['correction_dist'].mean()
cd_male   = df_clean[df_clean['gender'] == 'male']['correction_dist'].mean()
group_fairness = abs(cd_female - cd_male)

# set the threshold value
epsilon = 0.05

print("GROUP FAIRNESS ANALYSIS")
print(f"Mean Correction Distance — females: {cd_female:.4f}")
print(f"Mean Correction Distance — males:   {cd_male:.4f}")
print(f"Difference |C(female) - C(male)|:   {group_fairness:.4f}")
print(f"Epsilon threshold:                  {epsilon}")
print()
if group_fairness < epsilon:
    print("RESULT: GROUP FAIRNESS SATISFIED")
    print(f"The classifier treats males and females equally")
    print(f"(difference {group_fairness:.4f} < epsilon {epsilon})")
else:
    print("RESULT: GROUP FAIRNESS VIOLATED")
    print(f"The classifier is gender-biased")
    print(f"(difference {group_fairness:.4f} >= epsilon {epsilon})")

GROUP FAIRNESS ANALYSIS
Mean Correction Distance — females: 0.7640
Mean Correction Distance — males:   0.7918
Difference |C(female) - C(male)|:   0.0278
Epsilon threshold:                  0.05

RESULT: GROUP FAIRNESS SATISFIED
The classifier treats males and females equally
(difference 0.0278 < epsilon 0.05)


The model seems to **satisfy Group Fairness**. Specifically, the absolute difference between the Correction Distance of our two groups is 0.0278, lower than the defined threshold (0.05), allowing us to conclude that both males and females are equally treated by the model.  



---



**Jaccard Index and Individual Fairness**

In order to assess Individual Fairness, we first need to compute the Jaccard Index of the subjects in our sample, which basically is the ratio between the intersection and the union of the set of features of two individuals, A and B. The formula can be written as following:  

**J(A, B) = |A ∩ B| / |A ∪ B|**

In our scenario, two subjects are **blindly similar** if and only if:

**1.** They **only differ by gender**, namely, the only variable they are not sharing is the protected attribute "gender".

**2.** The Jaccard Index is **higher than 0.5**, ensuring a substantial overlap between their profiles.


Once obtained the Jaccard Index, we can implement the formula for Individual Fairness:

**Fairness_ind(S,T) = |C(S,GAD(a)) − C(S,GAD(b))| < ε**

where, for any pair male-female with Jaccard > 0.5, the formula verifies whether the absolute difference between their Correction Distance is less than a threshold ε, meaning that the model is individually fair for that specific pair.


In the code below:
* after defining the function for computing Jaccard
Index, we define get_feature_set() to convert numerical values of features like age into discrete categories (**bins**), so that we can group individuals with very similar characteristics and compare them.

* next, we **normalize NPR features** by converting all their values into a scale from 0 to 1, thus being sure that all variables will weigh the same in the comparison.  

* finally, we select random samples of **50 males and 50 females** (in order to simplify computations and keep at the same time a large number of comparisons, i.e. 50 x 50 = 2500, which allow us to preserve robust results) and proceed with the analysis of Blind Similarity  

In [ ]:
# INDIVIDUAL FAIRNESS THROUGH JACCARD INDEX

def jaccard_index(set_a, set_b):

    intersection = len(set_a.intersection(set_b))
    union = len(set_a.union(set_b))
    return intersection / union if union > 0 else 0

def get_feature_set(row, features, n_bins=5):

    feature_set = set()
    for f in features:
        val = row[f]
        # Create a descriptive string for each feature+value
        feature_set.add(f"{f}_bin{int(val * n_bins)}")
    return feature_set

# NPR features
npr_features = ['phq_score', 'pss_score', 'isi_score', 'age']

# Normalizing NPR features in [0,1] for the comparison
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df_clean_scaled = df_clean.copy()
df_clean_scaled[npr_features] = scaler.fit_transform(df_clean[npr_features])

# To simplify calculations, we select random samples with 50 females and 50 males
sample_female = df_clean_scaled[df_clean_scaled['gender'] == 'female'].sample(
    50, random_state=42)
sample_male   = df_clean_scaled[df_clean_scaled['gender'] == 'male'].sample(
    50, random_state=42)


# Find 'blindly similar' pairs and assess Individual Fairness
results_ind = []

for _, female_row in sample_female.iterrows():
    for _, male_row in sample_male.iterrows():

        # Calculating Jaccard Index on NPR features
        set_f = get_feature_set(female_row, npr_features)
        set_m = get_feature_set(male_row,   npr_features)
        jac   = jaccard_index(set_f, set_m)

        # Blind Similarity: Jaccard > 0.5 and only differ by gender
        if jac > 0.5:
            cd_f = female_row['correction_dist']
            cd_m = male_row['correction_dist']
            diff = abs(cd_f - cd_m)

            results_ind.append({
                'jaccard':      round(jac, 4),
                'cd_female':    round(cd_f, 4),
                'cd_male':      round(cd_m, 4),
                'cd_diff':      round(diff, 4),
                'fair':         diff < epsilon
            })

results_df = pd.DataFrame(results_ind)

print("\nINDIVIDUAL FAIRNESS ANALYSIS")
print(f"\nBlindly similar pairs found: {len(results_df)}")

if len(results_df) > 0:
    print(f"\nFairness satisfied in: "
          f"{results_df['fair'].sum()} / {len(results_df)} pairs "
          f"({results_df['fair'].mean()*100:.1f}%)")
    print(f"\nMean Jaccard Index:       {results_df['jaccard'].mean():.4f}")
    print(f"Mean CD difference:       {results_df['cd_diff'].mean():.4f}")

    print("\nMean Correction Distance for blindly similar pairs:")
    print(f"  Females: {results_df['cd_female'].mean():.4f}")
    print(f"  Males:   {results_df['cd_male'].mean():.4f}")

    # Investigating Individual Fairness violation
    results_df['favors'] = results_df.apply(
    lambda row: 'female' if row['cd_female'] < row['cd_male']
                else 'male' if row['cd_female'] > row['cd_male']
                else 'equal',
    axis=1
    )

    # Among violated pairs
    violated = results_df[results_df['fair'] == False]
    print("\nDirection of CD difference among violated pairs:")
    print(violated['favors'].value_counts())
    print(f"  % favoring females: "
          f"{(violated['favors'] == 'female').mean()*100:.1f}%")
    print(f"  % favoring males:   "
          f"{(violated['favors'] == 'male').mean()*100:.1f}%")

else:
    print("No blindly similar pairs found.")
    print("Try increasing the sample size or lowering the Jaccard threshold.")



INDIVIDUAL FAIRNESS ANALYSIS

Blindly similar pairs found: 611

Fairness satisfied in: 145 / 611 pairs (23.7%)

Mean Jaccard Index:       0.6589
Mean CD difference:       0.1994

Mean Correction Distance for blindly similar pairs:
  Females: 0.7277
  Males:   0.7893

Direction of CD difference among violated pairs:
favors
female    305
male      161
Name: count, dtype: int64
  % favoring females: 65.5%
  % favoring males:   34.5%


Out of 2500 pairs compared, 611 pairs were identified as blindly similar (Jaccard > 0.5), meaning they share sufficient NPR feature profiles while differing only in gender, the protected attribute.

Among these 611 blindly similar pairs, **Individual Fairness is satisfied in only 23.7% of cases**, meaning that in the vast majority of cases (76.3%), two clinically similar patients receive significantly different GAD predictions based merely on their gender.

Among the 466 pairs violating Individual Fairness, in 65.5%
of cases (305/466), the model assigns a lower correction
distance to the female patient, indicating higher predictive
confidence for females. This asymmetry confirms that
the Individual Fairness violation is not random but it is **systematically favoring female patients** in GAD prediction.

This result constitutes a clear **violation of Individual Fairness** and represents the central finding of this project: group fairness and individual fairness are not equivalent, and a classifier can satisfy one while violating the other.